# 04 - Pitcher fit, projected results, and free-agent targets

Each pitcher's mix of batted balls (by batter side and type) is weighted by each park's effects. Two different questions come out of that, and they should not be mixed up:

- **Fit** (`fit_pts`): how much better this park suits him than it suits a typical pitcher. It is *relative*. At a very hitter-friendly park every pitcher is hurt, so the best fit is just the pitcher who is hurt least.
- **Projected wOBA allowed** (`proj_woba`): his park-neutral wOBA allowed over the last two seasons (shrunk toward the league average) adjusted by this park's expected effect on him. It combines talent and park, so a lower number means he would actually pitch better there.

Other columns: `edge_pts` is the expected wOBA points on contact vs. a league-average park (positive helps the pitcher), and `talent_woba` is his park-neutral wOBA allowed.

Oakland Coliseum and Steinbrenner Field are left out of scoring because no team plays there now. Sutter Health Park has only 2 seasons of data, so it is kept but low-confidence (see `park_seasons`).

In [ ]:
import sys
sys.path.append("..")

import pandas as pd
from src.config import PROCESSED, INTERIM
from src.park_effects import contact_frame
from src.fit import pitcher_fit, pitcher_talent, project_park_woba, best_pitchers_for_park
from src.forecast import pitcher_seasons, next_season_pairs, calibrate_talent

df = pd.read_parquet(INTERIM / "pitches_archetyped.parquet")
contact = contact_frame(df)
effects = pd.read_csv(PROCESSED / "park_bb_effects.csv")

names = contact.drop_duplicates("pitcher").set_index("pitcher")["player_name"]
fit = pitcher_fit(contact, effects, min_bip=300)
# How much of a pitcher's contact luck should we take out, and how hard should we pull toward average?
# Fit that on past seasons (this season -> next season), judged on seasons it was not fitted on.
pairs = next_season_pairs(pitcher_seasons(df, effects), min_pa=200)
cal_summary, cal_folds, params = calibrate_talent(pairs)
cal_summary.round(2).to_csv(PROCESSED / "talent_calibration.csv", index=False)
pd.DataFrame([params]).to_csv(PROCESSED / "talent_params.csv", index=False)
print(f"{len(pairs)} pitcher-season pairs. Error predicting next season's park-neutral wOBA allowed (pts, lower is better):")
print(cal_summary.round(1).to_string(index=False))
print("Fitted:", params)

talent = pitcher_talent(df, effects, recent_seasons=2, shrink_pa=params["shrink_pa"], luck_weight=params["luck_weight"])
proj = project_park_woba(fit, talent)
proj["park_seasons"] = proj["park"].map(df.groupby("park")["game_year"].nunique())
print(f"{fit['pitcher'].nunique()} pitchers scored for fit, {len(talent)} with a talent estimate, {proj['pitcher'].nunique()} in both")
proj.head()

## Best-fit pitchers for a park

In [ ]:
best_pitchers_for_park(fit, "Coors Field", names)

In [ ]:
best_pitchers_for_park(fit, "Oracle Park", names)

## Free-agent class

`data/processed/free_agents_2026.csv` lists this winter's free-agent pitchers (columns `name`, `role`, `team_2026`, `notes`). Names use Statcast's "Last, First" format, and the notebook looks up each pitcher's ID from the data, ignoring accents. Edit the CSV as signings happen.

Pitchers without enough MLB data (for example, players coming from Japan or Korea) will not match and are listed below.

In [ ]:
import unicodedata

def key(s):
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode().lower()
    return " ".join(s.replace(".", "").replace(",", " ").split())

lookup = {key(n): p for p, n in names.items()}
fa = pd.read_csv(PROCESSED / "free_agents_2026.csv")
fa["mlbam_id"] = fa["name"].map(lambda n: lookup.get(key(n)))
print(f"Matched {fa['mlbam_id'].notna().sum()} of {len(fa)} free agents to Statcast IDs")
print("Not matched:", fa.loc[fa["mlbam_id"].isna(), "name"].tolist())

fa = fa.dropna(subset=["mlbam_id"]).astype({"mlbam_id": int})
fa_proj = proj.merge(fa, left_on="pitcher", right_on="mlbam_id")
print("No score (too few balls in play or recent plate appearances):", sorted(set(fa["name"]) - set(fa_proj["name"])))

## Who would actually pitch best at each park? (projected wOBA allowed)

The three free agents with the lowest projected wOBA allowed at each park. `park_seasons` shows how much data the park has; treat parks with fewer than 3 seasons as low-confidence.

In [ ]:
by_proj = (fa_proj.sort_values("proj_woba").groupby("park").head(3)
                  .sort_values(["park", "proj_woba"]))
cols = ["park", "park_seasons", "name", "role", "proj_woba", "talent_woba", "edge_pts", "fit_pts"]
by_proj[cols].round({"proj_woba": 3, "talent_woba": 3, "edge_pts": 1, "fit_pts": 1}).to_csv(
    PROCESSED / "fa_targets_by_park.csv", index=False)
print(by_proj[cols].round({"proj_woba": 3, "talent_woba": 3, "edge_pts": 1, "fit_pts": 1}).to_string(index=False))

## Buy-low: whose recent results look worse than the pitcher

A pitcher's raw results are distorted by the parks he worked in and by luck on contact. For each pitcher we split the gap between his recent wOBA allowed and his projection into parts (wOBA points, positive = better than his raw results):

- `past_park_pts`: removing the parks he pitched in (a hitter's park inflated his numbers)
- `regression_pts`: pulling extreme results toward the league average
- `new_park_pts`: the new park's expected effect on him (mostly the park-wide effect, which is validated; the pitcher-specific fit is a small extra)

`undervalued_pts` = `past_park_pts` + `regression_pts`: how much his raw results understate his talent, wherever he pitches. `contact_luck_pts` shows how much hitters beat the quality of contact (positive = unlucky for the pitcher; includes defense).

Contract cost is not in this data. Pair the list with a salary projection to find who is cheap relative to his talent.

In [ ]:
fa_proj["undervalued_pts"] = fa_proj["past_park_pts"] + fa_proj["regression_pts"]
rnd = {"raw_woba": 3, "neutral_woba": 3, "talent_woba": 3, "best_proj": 3, "undervalued_pts": 1, "contact_luck_pts": 1}

summary = (fa_proj.sort_values("proj_woba")
                  .groupby("name")
                  .agg(role=("role", "first"), pa=("pa", "first"), raw_woba=("raw_woba", "first"),
                       neutral_woba=("neutral_woba", "first"), talent_woba=("talent_woba", "first"),
                       undervalued_pts=("undervalued_pts", "first"), contact_luck_pts=("contact_luck_pts", "first"),
                       best_park=("park", "first"), best_proj=("proj_woba", "first"))
                  .sort_values("undervalued_pts", ascending=False)
                  .reset_index())
summary.round(rnd).to_csv(PROCESSED / "fa_buy_low_summary.csv", index=False)
print(summary.round(rnd).to_string(index=False))

In [ ]:
low = (fa_proj.sort_values("gain_pts", ascending=False).groupby("park").head(3)
              .sort_values(["park", "gain_pts"], ascending=[True, False]))
cols = ["park", "park_seasons", "name", "role", "pa", "raw_woba", "proj_woba", "gain_pts",
        "past_park_pts", "regression_pts", "new_park_pts"]
low[cols].round({"raw_woba": 3, "proj_woba": 3, "gain_pts": 1, "past_park_pts": 1, "regression_pts": 1, "new_park_pts": 1}) \
    .to_csv(PROCESSED / "fa_buy_low_by_park.csv", index=False)
print(low[cols].round({"raw_woba": 3, "proj_woba": 3, "gain_pts": 1, "past_park_pts": 1,
                       "regression_pts": 1, "new_park_pts": 1}).to_string(index=False))

### Buy-low across the whole league (not just free agents)

Pitchers with 300+ plate appearances in the last two seasons, ranked by how much their raw results understate them. This includes pitchers who are not free agents, so it also works as a trade-target list.

In [ ]:
league = (proj.drop_duplicates("pitcher")
              .assign(name=lambda d: d["pitcher"].map(names),
                      undervalued_pts=lambda d: d["past_park_pts"] + d["regression_pts"])
              .query("pa >= 300")
              .sort_values("undervalued_pts", ascending=False)
              .head(40))
out = league[["name", "pa", "raw_woba", "neutral_woba", "talent_woba", "undervalued_pts", "contact_luck_pts"]]
out = out.round({"raw_woba": 3, "neutral_woba": 3, "talent_woba": 3, "undervalued_pts": 1, "contact_luck_pts": 1})
out.to_csv(PROCESSED / "buy_low_league.csv", index=False)
print(out.to_string(index=False))

## Best park fit, regardless of talent

The three free agents whose profile suits each park best compared with a typical pitcher. This says nothing about how good they are.

In [ ]:
by_fit = (fa_proj.sort_values("fit_pts", ascending=False).groupby("park").head(3)
                 .sort_values(["park", "fit_pts"], ascending=[True, False]))
cols = ["park", "park_seasons", "name", "role", "fit_pts", "edge_pts"]
by_fit[cols].round(1).to_csv(PROCESSED / "fa_best_fit_by_park.csv", index=False)
print(by_fit[cols].round(1).to_string(index=False))

## Best and worst park for each free agent (by fit)

In [ ]:
bw = (fa_proj.sort_values("fit_pts")
             .groupby("name")
             .agg(role=("role", "first"), worst_park=("park", "first"), worst=("fit_pts", "first"),
                  best_park=("park", "last"), best=("fit_pts", "last"))
             .round(1)
             .sort_values("best", ascending=False))
print(bw.to_string())

## Next

The backtest in `05_backtest.ipynb` checks whether these park effects predict real results.

## Commit

```
git add notebooks/04_fa_targets.ipynb data/processed
git commit -m "pitcher fit, projected wOBA allowed, free-agent targets"
git push
```